# Phase 5: Swap pipes for network sockets

Same topology as Phase 4 (Parent -> Worker 1 -> Worker 2 -> Parent) but control and data go over TCP sockets instead of pipes + shared memory.

In [1]:
import os
import sys
import pickle
import socket
import struct
import subprocess
import time
import zlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "Hey! How are you feeling today?"
k = 2
num_middle_partitions = 2
num_new_tokens = 16

PORT_PARENT = 5100
PORT_W1 = 5101
PORT_W2 = 5102

MAGIC = b"P5CH"
VERSION = 1
KIND_HELLO = "hello"
KIND_READY = "ready"
KIND_TENSOR = "tensor"
KIND_SHUTDOWN = "shutdown"
KIND_ERROR = "error"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:
base = getattr(model, "model", model)
embed_module = getattr(base, "embed_tokens", None) or getattr(base, "embed", None)
layers_list = getattr(base, "layers", None)
final_norm = getattr(base, "norm", None)
lm_head_module = getattr(model, "lm_head", None)
rotary_emb = getattr(base, "rotary_emb", None)
assert embed_module is not None and layers_list is not None
assert final_norm is not None and lm_head_module is not None and rotary_emb is not None

num_blocks = len(layers_list)
first_k_layers = layers_list[:k]
last_k_layers = layers_list[-k:]
middle_start = k
middle_end = num_blocks - k
middle_count = middle_end - middle_start
assert middle_count > 0 and middle_count % num_middle_partitions == 0
chunk = middle_count // num_middle_partitions
ranges = [(middle_start + i * chunk, middle_start + (i + 1) * chunk) for i in range(num_middle_partitions)]

HIDDEN_SIZE = model.config.hidden_size
MAX_SEQ = 2048
BATCH_SIZE = 1
max_nbytes = BATCH_SIZE * MAX_SEQ * HIDDEN_SIZE * 4

print(f"total blocks={num_blocks}, first_k={k}, middle_count={middle_count}, last_k={k}")
print("middle partition ranges:", ranges)

total blocks=24, first_k=2, middle_count=20, last_k=2
middle partition ranges: [(2, 12), (12, 22)]


In [3]:
def _configure_sock(sock, timeout_s=120.0):
    sock.settimeout(timeout_s)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_KEEPALIVE, 1)


def _send_frame(sock, frame, tensor_bytes=None):
    payload = pickle.dumps(frame, protocol=pickle.HIGHEST_PROTOCOL)
    tensor_bytes = tensor_bytes or b""
    checksum = zlib.crc32(payload) & 0xFFFFFFFF
    header = struct.pack(
        ">4sBIII",
        MAGIC,
        VERSION,
        len(payload),
        len(tensor_bytes),
        checksum,
    )
    sock.sendall(header)
    sock.sendall(payload)
    if tensor_bytes:
        sock.sendall(tensor_bytes)


def _recv_exact(sock, n):
    buf = []
    while n > 0:
        chunk = sock.recv(n)
        if not chunk:
            raise ConnectionError("socket closed")
        buf.append(chunk)
        n -= len(chunk)
    return b"".join(buf)


def _recv_frame(sock, max_nbytes):
    header = _recv_exact(sock, 17)
    magic, version, payload_len, tensor_len, checksum = struct.unpack(">4sBIII", header)
    if magic != MAGIC:
        raise ValueError(f"Bad frame magic: {magic!r}")
    if version != VERSION:
        raise ValueError(f"Unsupported frame version: {version}")
    if payload_len > 1_000_000:
        raise ValueError(f"payload too large: {payload_len}")
    if tensor_len > max_nbytes:
        raise ValueError(f"tensor_len={tensor_len} > max_nbytes={max_nbytes}")

    payload = _recv_exact(sock, payload_len)
    if (zlib.crc32(payload) & 0xFFFFFFFF) != checksum:
        raise ValueError("Frame checksum mismatch")

    frame = pickle.loads(payload)
    tensor_bytes = _recv_exact(sock, tensor_len) if tensor_len else None
    return frame, tensor_bytes


def _send_hello_expect_ready(sock, role):
    _send_frame(sock, {"kind": KIND_HELLO, "role": role})
    frame, _ = _recv_frame(sock, max_nbytes)
    if frame.get("kind") != KIND_READY:
        raise RuntimeError(f"Expected READY, got {frame}")


def _expect_hello_send_ready(sock, role):
    frame, _ = _recv_frame(sock, max_nbytes)
    if frame.get("kind") != KIND_HELLO:
        raise RuntimeError(f"Expected HELLO, got {frame}")
    _send_frame(sock, {"kind": KIND_READY, "role": role})


def run_layers(hidden_states, layers):
    seq_len = hidden_states.shape[1]
    position_ids = torch.arange(seq_len, device=DEVICE, dtype=torch.long).unsqueeze(0)
    position_embeddings = rotary_emb(hidden_states, position_ids)
    causal_mask = torch.triu(
        torch.full((seq_len, seq_len), torch.finfo(hidden_states.dtype).min, device=DEVICE, dtype=hidden_states.dtype),
        diagonal=1,
    ).unsqueeze(0).unsqueeze(0)
    for layer in layers:
        hidden_states = layer(
            hidden_states,
            attention_mask=causal_mask,
            position_embeddings=position_embeddings,
            position_ids=position_ids,
            use_cache=False,
        )
    return hidden_states


def get_baseline(prompt, num_tokens):
    inp = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = inp["input_ids"].clone()
    with torch.no_grad():
        for _ in range(num_tokens):
            out = model(input_ids=input_ids, use_cache=False)
            next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_id], dim=-1)
    return input_ids[0, inp["input_ids"].shape[1]:].tolist()

In [4]:
worker_script = os.path.join(os.getcwd(), "middle_worker_v5.py")
assert num_middle_partitions == 2


def _read_worker_failure(proc, name):
    stderr = ""
    try:
        _, err = proc.communicate(timeout=1)
        stderr = (err or "").decode("utf-8", errors="replace") if isinstance(err, bytes) else (err or "")
    except Exception:
        pass
    raise RuntimeError(f"{name} exited with code {proc.returncode}. stderr:\n{stderr}")


def _connect_parent_to_w1_with_retry(w1, w2, timeout_s=120.0):
    deadline = time.monotonic() + timeout_s
    sleep_s = 0.05
    last_error = None
    while time.monotonic() < deadline:
        if w1.poll() is not None:
            _read_worker_failure(w1, "W1")
        if w2.poll() is not None:
            _read_worker_failure(w2, "W2")

        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        _configure_sock(sock)
        try:
            sock.connect(("127.0.0.1", PORT_W1))
            return sock
        except OSError as exc:
            last_error = exc
            sock.close()
            time.sleep(sleep_s)
            sleep_s = min(1.0, sleep_s * 2)

    raise TimeoutError(f"Timed out connecting parent->W1; last_error={last_error}")


def _accept_w2_with_supervision(listen_sock, w1, w2, timeout_s=120.0):
    deadline = time.monotonic() + timeout_s
    listen_sock.settimeout(1.0)
    while time.monotonic() < deadline:
        if w1.poll() is not None:
            _read_worker_failure(w1, "W1")
        if w2.poll() is not None:
            _read_worker_failure(w2, "W2")
        try:
            sock, _ = listen_sock.accept()
            _configure_sock(sock)
            return sock
        except socket.timeout:
            continue
    raise TimeoutError("Timed out waiting for W2->parent connection")


def start_workers():
    listen_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    listen_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    listen_sock.bind(("", PORT_PARENT))
    listen_sock.listen(1)

    w2 = subprocess.Popen(
        [sys.executable, worker_script, MODEL_NAME, str(PORT_W2), "127.0.0.1", str(PORT_PARENT), str(max_nbytes), str(ranges[1][0]), str(ranges[1][1])],
        cwd=os.getcwd(),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    w1 = subprocess.Popen(
        [sys.executable, worker_script, MODEL_NAME, str(PORT_W1), "127.0.0.1", str(PORT_W2), str(max_nbytes), str(ranges[0][0]), str(ranges[0][1])],
        cwd=os.getcwd(),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    sock_to_w1 = None
    sock_from_w2 = None
    try:
        sock_to_w1 = _connect_parent_to_w1_with_retry(w1, w2)
        sock_from_w2 = _accept_w2_with_supervision(listen_sock, w1, w2)
        _expect_hello_send_ready(sock_from_w2, role="parent")
        _send_hello_expect_ready(sock_to_w1, role="parent")
        return (sock_to_w1, sock_from_w2), (w1, w2)
    except Exception:
        if sock_to_w1 is not None:
            sock_to_w1.close()
        if sock_from_w2 is not None:
            sock_from_w2.close()
        for p in (w1, w2):
            if p.poll() is None:
                p.terminate()
                try:
                    p.wait(timeout=5)
                except Exception:
                    p.kill()
        raise
    finally:
        listen_sock.close()


def run_chain_inference(socks, prompt, num_tokens):
    sock_to_w1, sock_from_w2 = socks

    def send_recv(hidden_states):
        t = hidden_states.cpu().float()
        nbytes = t.numel() * t.element_size()
        if nbytes > max_nbytes:
            raise ValueError(f"nbytes={nbytes} > max_nbytes={max_nbytes}")
        _send_frame(
            sock_to_w1,
            {"kind": KIND_TENSOR, "shape": tuple(t.shape), "nbytes": nbytes},
            t.numpy().tobytes(),
        )
        frame, out_bytes = _recv_frame(sock_from_w2, max_nbytes)
        kind = frame.get("kind")
        if kind == KIND_ERROR:
            raise RuntimeError(f"Worker error: {frame.get('message')}")
        if kind != KIND_TENSOR:
            raise RuntimeError(f"Unexpected response frame: {frame}")
        out_shape = tuple(frame["shape"])
        return torch.frombuffer(out_bytes, dtype=torch.float32).reshape(out_shape).clone().to(DEVICE)

    inp = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = inp["input_ids"].clone()
    with torch.no_grad():
        for _ in range(num_tokens):
            hidden = embed_module(input_ids)
            hidden = run_layers(hidden, first_k_layers)
            hidden = send_recv(hidden)
            hidden = run_layers(hidden, last_k_layers)
            hidden = final_norm(hidden)
            logits = lm_head_module(hidden[:, -1:, :])
            next_id = logits.argmax(dim=-1)
            input_ids = torch.cat([input_ids, next_id], dim=-1)
    return input_ids[0, inp["input_ids"].shape[1]:].tolist()


def shutdown_workers(socks, procs):
    sock_to_w1, sock_from_w2 = socks
    _send_frame(sock_to_w1, {"kind": KIND_SHUTDOWN})
    frame, _ = _recv_frame(sock_from_w2, max_nbytes)
    assert frame.get("kind") == KIND_SHUTDOWN
    sock_to_w1.close()
    sock_from_w2.close()
    procs[0].wait()
    procs[1].wait()
    if procs[0].returncode != 0 or procs[1].returncode != 0:
        _read_worker_failure(procs[0], "W1") if procs[0].returncode != 0 else _read_worker_failure(procs[1], "W2")

In [5]:
generated_ids_baseline = get_baseline(PROMPT, num_new_tokens)
decoded_baseline = tokenizer.decode(generated_ids_baseline, skip_special_tokens=True)
print("Baseline:", generated_ids_baseline)
print("Baseline decoded:", decoded_baseline)

Baseline: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"


In [6]:
socks, procs = start_workers()
generated_ids_split = run_chain_inference(socks, PROMPT, num_new_tokens)
shutdown_workers(socks, procs)
decoded_split = tokenizer.decode(generated_ids_split, skip_special_tokens=True)
print("Split:", generated_ids_split)
print("Split decoded:", decoded_split)

/var/folders/21/yff35t8977n9_7z54_9jjqp40000gn/T/ipykernel_35001/473766008.py:120: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:1587.)
  return torch.frombuffer(out_bytes, dtype=torch.float32).reshape(out_shape).clone().to(DEVICE)


Split: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Split decoded:  Are you feeling good? I'm not sure what you mean by "good"


In [7]:
match = generated_ids_baseline == generated_ids_split
print("Token-by-token comparison:")
for i in range(num_new_tokens):
    status = "ok" if generated_ids_baseline[i] == generated_ids_split[i] else "MISMATCH"
    print(f"  {i}: baseline={generated_ids_baseline[i]}, split={generated_ids_split[i]} [{status}]")
print("\nPASS" if match else "FAIL")
print("\nBaseline decoded:", decoded_baseline)
print("Split decoded:    ", decoded_split)

Token-by-token comparison:
  0: baseline=8713, split=8713 [ok]
  1: baseline=498, split=498 [ok]
  2: baseline=8266, split=8266 [ok]
  3: baseline=1661, split=1661 [ok]
  4: baseline=30, split=30 [ok]
  5: baseline=358, split=358 [ok]
  6: baseline=2776, split=2776 [ok]
  7: baseline=537, split=537 [ok]
  8: baseline=2704, split=2704 [ok]
  9: baseline=1128, split=1128 [ok]
  10: baseline=498, split=498 [ok]
  11: baseline=3076, split=3076 [ok]
  12: baseline=553, split=553 [ok]
  13: baseline=330, split=330 [ok]
  14: baseline=18536, split=18536 [ok]
  15: baseline=1, split=1 [ok]

PASS

Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"
Split decoded:      Are you feeling good? I'm not sure what you mean by "good"


In [8]:
PROMPT2 = "What is 2+2?"
num_tokens2 = 8
baseline2 = get_baseline(PROMPT2, num_tokens2)
socks2, procs2 = start_workers()
split2 = run_chain_inference(socks2, PROMPT2, num_tokens2)
shutdown_workers(socks2, procs2)
match2 = baseline2 == split2
print("Test 2 (different prompt, different length):", "PASS" if match2 else "FAIL")
if not match2:
    print("  baseline:", baseline2)
    print("  split:   ", split2)

Test 2 (different prompt, different length): PASS


In [9]:
num_tokens_long = 64
baseline_long = get_baseline(PROMPT, num_tokens_long)
socks_long, procs_long = start_workers()
split_long = run_chain_inference(socks_long, PROMPT, num_tokens_long)
shutdown_workers(socks_long, procs_long)
match_long = baseline_long == split_long
print("Test 3 (longer generation, 64 tokens):", "PASS" if match_long else "FAIL")
if not match_long:
    for i in range(num_tokens_long):
        if baseline_long[i] != split_long[i]:
            print(f"  First mismatch at index {i}: baseline={baseline_long[i]}, split={split_long[i]}")
            break

Test 3 (longer generation, 64 tokens): PASS
